# Play Colour Bomb Gridworlds

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play with the arrow keys or WASD in a pygame window. Set `RECORD_VIDEO = True` to save the environment canvas as a 1920x1080 MP4.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from masa.common.rendering.notebook_play import make_reset_env, sync_selected_env
from masa.common.rendering.notebook_play import start_recorded_play_thread

VALID_ENV_NAMES = (
    "ColourBombGridWorld",
    "ColourBombGridWorldV2",
    "ColourBombGridWorldV3",
)


def make_env(env_name, **kwargs):
    if env_name == "ColourBombGridWorld":
        from masa.envs.tabular.colour_bomb_grid_world import ColourBombGridWorld

        return ColourBombGridWorld(**kwargs)
    if env_name == "ColourBombGridWorldV2":
        from masa.envs.tabular.colour_bomb_grid_world_v2 import ColourBombGridWorldV2

        return ColourBombGridWorldV2(**kwargs)
    if env_name == "ColourBombGridWorldV3":
        from masa.envs.tabular.colour_bomb_grid_world_v3 import ColourBombGridWorldV3

        return ColourBombGridWorldV3(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")


ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES, value="ColourBombGridWorld", description="Env"
)
display(ENV_SELECTOR)
ENV_NAME = ENV_SELECTOR.value
SEED = 0

RECORD_VIDEO = False
VIDEO_PATH = "videos/notebooks/play_colour_bomb_gridworlds.mp4"

In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(ENV_NAME, render_mode="rgb_array", render_window_size=512)
obs, info = preview_env.reset(seed=SEED)
print("reset:", {"obs": obs, "info": info})
display(Image.fromarray(preview_env.render()))
preview_env.close()

In [ ]:
def _print_reset(obs, info):
    print("reset:", {"obs": obs, "info": info})


def play_env(env_name=None, seed=SEED):
    follow_selector = env_name is None

    def _run(stop_event):
        import pygame

        selected_env_name = env_name
        if follow_selector:
            selected_env_name = ENV_SELECTOR.value
        action_keys = {
            pygame.K_LEFT: 0,
            pygame.K_a: 0,
            pygame.K_RIGHT: 1,
            pygame.K_d: 1,
            pygame.K_DOWN: 2,
            pygame.K_s: 2,
            pygame.K_UP: 3,
            pygame.K_w: 3,
            pygame.K_SPACE: 4,
        }
        env, obs, info = make_reset_env(
            make_env,
            selected_env_name,
            seed=seed,
            render_mode="human",
            render_window_size=512,
        )
        finished = False
        print(
            "Controls: arrows or WASD move, Space waits, R resets, Q or Escape quits."
        )
        _print_reset(obs, info)
        try:
            running = True
            while running and not stop_event.is_set() and not env.human_window_closed:
                if follow_selector:
                    env, selected_env_name, obs, info, switched = sync_selected_env(
                        env,
                        selected_env_name,
                        ENV_SELECTOR,
                        make_env,
                        seed=seed,
                        render_window_size=512,
                        pygame=pygame,
                    )
                    if switched:
                        finished = False
                        print("switched:", selected_env_name)
                        _print_reset(obs, info)
                action = None
                for event in pygame.event.get():
                    if not env.handle_pygame_event(event):
                        running = False
                        break
                    if event.type != pygame.KEYDOWN:
                        continue
                    if event.key in (pygame.K_q, pygame.K_ESCAPE):
                        running = False
                        break
                    if event.key == pygame.K_r:
                        obs, info = env.reset(seed=seed)
                        finished = False
                        _print_reset(obs, info)
                        continue
                    if event.key in action_keys and not finished:
                        action = action_keys[event.key]
                if not running or stop_event.is_set():
                    break
                if action is None:
                    env.render()
                    continue
                obs, reward, terminated, truncated, info = env.step(action)
                finished = terminated or truncated
                print(
                    {
                        "obs": obs,
                        "reward": reward,
                        "terminated": terminated,
                        "truncated": truncated,
                        "info": info,
                    }
                )
        finally:
            env.close()

    return start_recorded_play_thread(
        "colour_bomb_gridworlds", _run, record_video=RECORD_VIDEO, video_path=VIDEO_PATH
    )


play_session = play_env(seed=SEED)